In [1]:
import pandas as pd 
import numpy as np 
import numpy as np
from scipy.spatial import procrustes
from sklearn.cluster import AgglomerativeClustering
from sklearn import metrics as skmetrics
import numpy as np
import pandas as pd
import itertools
import pickle
import sys
import os
import shutil
import tempfile
import subprocess
# redefining some functions from simulate.py 

def mean_squared_error(original, recovered, mask = None):
    if mask is None: mask = np.ones_like(original)
    n = np.sum(mask)
    mse = np.sum(np.square((original - recovered) * mask)) / n
    return mse


def peak_signal_to_noise_ratio(original, recovered, mask = None):
    if mask is None: mask = np.ones_like(original)
    omax = np.max(original[mask == 1])
    omin = np.min(original[mask == 1])
    maxsig2 = np.square(omax - omin)
    mse = mean_squared_error(original, recovered, mask)
    res = 10 * np.log10(maxsig2 / mse)
    return res


def matrix_dissimilarity_scores(original, recovered, mask = None, match = 'zerofill'):
    '''
    Procrustes analysis returns the square of the Frobenius norm.
    Use the rotated matrix to obtain the peak signal-to-noise ratio (PSNR).
    Input matrices can have different dimensions.
    There are two ways to match:
        - clip: remove information from the larger matrix
        - zerofill: pad zero columns in the smaller matrix
    '''
    n_orig = original.shape[1]
    n_recv = recovered.shape[1]
    m = original.shape[0]
    if match == 'clip':
        n = min(n_orig, n_recv)
        X = original[:, :n]
        Y = recovered[:, :n]
    elif match == 'zerofill':
        n = max(n_orig, n_recv)
        X = np.zeros((m, n))
        Y = np.zeros((m, n))
        X[:, :n_orig] = original
        Y[:, :n_recv] = recovered
    R_orig, R_recv, m2 = procrustes(X, Y)
    psnr = peak_signal_to_noise_ratio(R_orig, R_recv, mask)
    return np.sqrt(m2), psnr


def adjusted_mutual_information_score(X, class_labels):
    X_cent = X - np.mean(X, axis = 0, keepdims = True)
    distance_matrix = skmetrics.pairwise.pairwise_distances(X_cent, metric='euclidean')
    model = AgglomerativeClustering(n_clusters = 5, linkage = 'average', metric = 'precomputed')
    class_pred = model.fit_predict(distance_matrix)
    return skmetrics.adjusted_mutual_info_score(class_labels, class_pred)


def distribute_samples_to_classes(Q, n, shuffle = False):
    '''
    Distribute n samples to Q classes
    '''
    rs = 0.6 * np.random.rand(Q) + 0.2 # random sample from [0.2, 0.8)
    z = np.array(np.round((rs / np.sum(rs)) * n), dtype = int)
    z[-1] = n - np.sum(z[:-1])
    tidx = np.arange(n)
    if shuffle:
        np.random.shuffle(tidx)
    bins = np.zeros(Q + 1, dtype = int)
    bins[1:] = np.cumsum(z)
    idx_groups  = [np.sort(tidx[bins[i]:bins[i+1]]) for i in range(Q)]
    labels = [i for idx in range(n) for i in range(Q) if idx in idx_groups[i]]
    return idx_groups, labels



def get_blockdiag_matrix(n, rholist, rhobg, idx_groups):
    '''
    Generate a block diagonal matrix of size n x n.
    S_ij = 1, if i = j
         = rholist[q],  if i,j \in idx_groups[q]
         = rhobg, otherwise
    '''
    R = np.ones((n, n)) * rhobg

    for i, (idx, rho) in enumerate(zip(idx_groups, rholist)):
        nblock = idx.shape[0]
        xblock = np.ones((nblock, nblock)) * rho
        R[np.ix_(idx, idx)] = xblock

    R[np.diag_indices_from(R)] = 1.0

    return R

# redefined simulate func 
def effect_size_gleanr(n, p, k, Q, h2, g2,
        aq, a0, nsample, 
        cov_design = 'blockdiag',
        sharing_proportion = 1.0,
        shuffle = False,
        seed = None):
    '''
    Get Y = LF' + M + E  where columns of F are orthonormal,
    and L is a blockdiagonal matrix.
    LF' correspond to the shared component of effect sizes,
    the distinct components are given by M, which is sampled
    from a Laplace distribution.
    The noise in the estimate of the effect sizes is given by E.
    '''
    if seed is not None: np.random.seed(seed)
    if not isinstance(h2, np.ndarray):
        h2 = np.ones(n) * h2
    if not isinstance(g2, np.ndarray):
        g2 = np.ones(n) * g2
    if not isinstance(nsample, np.ndarray):
        nsample = np.ones(n) * nsample
    p_shared = int(p * sharing_proportion)

    C_ixgrp, C = distribute_samples_to_classes(Q, n, shuffle = shuffle)
    ggT  = np.sqrt(np.einsum('i,j->ij', g2, g2))
    if cov_design == 'blockdiag':
        rho  = [aq for _ in range(Q)]
        covL = get_blockdiag_matrix(n, rho, a0, C_ixgrp) * ggT
    else:
        covL = np.eye(n) * ggT
    # normalize L for correct variance.
    L  = np.random.multivariate_normal(np.zeros(n), covL, size = k).T 
    L /= np.sqrt(k)
    # F with orthogonal columns
    F  = spstats.ortho_group.rvs(p_shared)[:, :k]
    # M is a sparse matrix of effect sizes
    scaleM = np.sqrt((h2 - g2) * 0.5 / p)
    M = np.random.laplace(np.zeros(n), scaleM, size = (p, n)).T
    # obtain the true effect sizes
    Y = np.zeros((n, p))
    p_choose = np.sort(np.random.choice(p, p_shared, replace = False))
    Y[:, p_choose] = L @ F.T
    Y += M
    # the observed effect size is normally distributed, with mean Y
    # and variance obtained from the residuals.
    stderr = np.sqrt((1 - np.square(Y)) / nsample.reshape(n, 1))
    Yobs = np.random.normal(Y, stderr)
    # observed Z-scores
    Z = Yobs / stderr
    cormat = np.corrcoef(Z)
    return Z, Yobs, Y, L, F, M, C, stderr,cormat

def generate_masked_input(Y, mask):
    Ymiss_nan = Y.copy()
    Ymiss_nan[mask] = np.nan
    Ymiss_nan_cent = Ymiss_nan - np.nanmean(Ymiss_nan, axis = 0, keepdims = True)
    # Ymiss_nan_cent[mask] = 0.0
    return Ymiss_nan_cent


def generate_mask(n, p, ratio):
    mask = np.ones(n * p)
    nzero = int(ratio * n * p)
    mask[:nzero] = 0.0
    np.random.shuffle(mask)
    return mask.reshape(n,p) == 0.



In [ ]:
gleanr_dir='gleanr_sims'
os.mkdir(gleanr_dir)

In [2]:
from scipy import stats as spstats
n = 200
p = 2000
k = 10
Q = 3
h2 = 0.2
h2_shared_frac = 0.5
aq = 0.6
a0 = 0.2
nsample_minmax = (10000, 40000)
sharing_proportion = 1.0
reps = 10
nsample = np.random.uniform(nsample_minmax[0], nsample_minmax[1], n)
g2 = h2 * h2_shared_frac

Z, effect_size_obs, effect_size_true, L, F, M, C,stderr,cormat = \
    effect_size_gleanr(
        n, p, k, Q, h2, g2, aq, a0, nsample,
        sharing_proportion = sharing_proportion,
        cov_design = 'blockdiag', shuffle = False,
        seed = None)

In [3]:
Z.shape

(200, 2000)

In [69]:
## {'p': 2000, 'k': 10, 'h2': 0.2, 'h2_shared_frac': 0.5, 'aq': 0.6}
# this is a placeholder for simulations 

from scipy import stats as spstats
n = 200
p = 2000
k = 10
Q = 3
h2 = 0.2
h2_shared_frac = 0.5
aq = 0.6
a0 = 0.2
nsample_minmax = (10000, 40000)
sharing_proportion = 1.0
reps = 10
nsample = np.random.uniform(nsample_minmax[0], nsample_minmax[1], n)
g2 = h2 * h2_shared_frac
for rep in range(reps):
    if os.path.isdir(gleanr_dir + f'/repl_{rep+1}/'):
        shutil.rmtree(gleanr_dir + f'/repl_{rep+1}/')
    os.mkdir(gleanr_dir + f'/repl_{rep+1}/')
    odir = gleanr_dir + f'/repl_{rep+1}/'
    Z, effect_size_obs, effect_size_true, L, F, M, C,stderr,cormat = \
        effect_size_gleanr(
            n, p, k, Q, h2, g2, aq, a0, nsample,
            sharing_proportion = sharing_proportion,
            cov_design = 'blockdiag', shuffle = False,
            seed = None)
    # column of names (gleanr req)
    effect_size_obs = pd.DataFrame(np.transpose(effect_size_obs),index=[f'rs{i+1}' for i in range(p)])
    effect_size_obs.columns = [f'trait_{i+1}' for i in range(200)]
    stderr = pd.DataFrame(np.transpose(stderr),index=[f'rs{i+1}' for i in range(p)])
    stderr.columns = [f'trait_{i+1}' for i in range(200)]

    cormat=pd.DataFrame(np.corrcoef(Z)) # np.corrcoef works row first for correlation operations. not the standard 
    cormat.columns = stderr.columns
    effect_size_obs.to_csv(f'{odir}sim_h2_{h2}_p_{p}_k_{k}_rep_{rep+1}_effect_size_obs.txt')
    stderr.to_csv(f'{odir}sim_h2_{h2}_p_{p}_k_{k}_rep_{rep+1}_effect_size_stderr.txt')
    cormat.round(4).to_csv(f'{odir}sim_h2_{h2}_p_{p}_k_{k}_rep_{rep+1}_cormat.txt',index=False,sep=' ') 
    # the round function is to prevent any floating point issues - the positive definite check in gleanr has a low tolerance but this 
# negates that issue 

with open("dummy_names.txt",'w') as w:
    for i in range(n):
        w.write('trait_' + str(i+1) + '\n')

In [95]:
# run basic gleanr - one replicate directory as an example
# saikat - this will probably change based on the dsc output save format 
import glob
scriptdir = '/Users/oconns04/Downloads/lrma-dsc-main/dsc/functions/' # this of course changes
rep_index = 1 # for example. looping is arbitrary
files = glob.glob(f'{gleanr_dir}/repl_{rep_index}/*.txt')
outdir = f'{gleanr_dir}/repl_{rep_index}/gleanr_out/'
if os.path.isdir(outdir):
        shutil.rmtree(outdir)
os.mkdir(outdir)
cmat = [i for i in files if 'cormat' in i]
stderr = [i for i in files if 'stderr' in i]
effs = [i for i in files if 'obs' in i]
cmd = f'Rscript {scriptdir}gleanr_run.R ' 
cmd += f'--gwas_effects {effs[0]} '
cmd += f'--uncertainty {stderr[0]} ' 
cmd += f'--fixed_first -K 6 -v 1 --covar_matrix {cmat[0]} '
cmd += f'--WLgamma 0.5 --trait_names {scriptdir}dummy_names.txt'
cmd += f' --outdir {outdir}'
# run script 

process = os.system(cmd)

Loading required package: usethis
Running GLEANR, with settings as follows:

------------------------------ INPUT FILES ------------------------------
Effect sizes: sim_h2_0.2_p_2000_k_10_rep_1_effect_size_obs.txt
Uncertainty estimates: sim_h2_0.2_p_2000_k_10_rep_1_effect_size_stderr.txt
Cohort overlap adjustment: sim_h2_0.2_p_2000_k_10_rep_1_cormat.txt
       Block distance: 0.2
       Shrinkage factor: 0.5
Genomic correction terms: 
Z-score sample standard deviation: 

------------------------------ INPUT SETTINGS ------------------------------
BIC convergence criteria: BIC.change
BIC method: sklearn_eBIC
K init: 6
------------------------------ OUTPUT SETTINGS ------------------------------
Output directory: gleanr_sims/repl_1/gleanr_out/

------------------------------ INPUT FILE PROCESSING ------------------------------
Standardizing explanatory variables (W_c(W_s*B)^T) by default
Using the provided trait names, and assuming all files have columns in the correct order.
It is the u

  |======================================================================| 100%


Fitting V, iteration 1


  |======================================================================| 100%


Fitting U, iteration 2


...

Iter11:
Proportion objective change = 0.001
Frobenius norm of (updated factor matrix - previous factor matrix) / number of factors  = 1.18133969740129
U Sparsity = 0.167; V sparsity = 0.206; 4 factors remain
Memory usage at the end of the current iteration: 165199960
Beginning on iteration 12
Now fitting V...


  |======================================================================| 100%


Now fitting U...


  |======================================================================| 100%


Current objective change: 9e-04
Objective function change threshold achieved!
Objective function converged at iteration 12
Total time used for model fitting: 0.326 min


In [96]:
# clean up 
outfiles = glob.glob(f'{outdir}*')
lf = pd.read_csv([i for i in outfiles if 'factors' in i][0],sep = ' ',index_col=0) # factors for phenos
ll = pd.read_csv([i for i in outfiles if 'loadings' in i][0],sep = ' ',index_col=0)#snp loadings for factors
lf

,V1,V2,V3,V4
Study,,,,
trait_1,-0.248643,6.819366,3.091104,0.000000
trait_2,-2.019623,1.139485,-1.136937,3.805181
trait_3,-2.072114,2.951065,3.341115,0.000000
trait_4,-1.018139,2.844442,0.000000,0.834240
trait_5,-3.382594,1.629852,0.000000,1.321041
...,...,...,...,...
trait_196,4.241391,0.000000,4.646083,0.000000
trait_197,6.966141,0.792370,-3.013029,-0.062467
trait_198,5.029987,0.853742,-1.550636,0.594043


In [97]:
ll

,U1,U2,U3,U4
SNP,,,,
rs999,-0.000162,-0.000051,-0.000386,0.000443
rs998,0.000343,-0.000213,0.000000,0.000000
rs997,0.000342,-0.000070,0.000000,0.000000
rs996,-0.000432,0.000322,-0.000292,-0.000839
rs995,-0.000345,0.000000,0.000000,-0.000013
...,...,...,...,...
rs1001,-0.000628,0.000255,0.000000,0.000588
rs1000,0.000041,0.000131,0.000056,0.000000
rs100,-0.000754,-0.000052,-0.000581,0.000033


gleanr(beta_m,W_s, snp_names, trait_names, C=c.mat, covar_se=c_se.mat, K="GRID",conv_objective=0.005, verbosity=0, save_out=FALSE)

effect_sizes (PxN)
std_err (PxN)
covar_matrix (NxN)
WLgamma


gleanr(effect_sizes, C=covar_matrix, K=6, fixed_ubiq=TRUE, shrinkWL=0.5)

In [ ]:
cmd = f'Rscript {scriptdir}gleanr_run.R ' 
cmd += f'--gwas_effects {effs[0]} '
cmd += f'--uncertainty {stderr[0]} ' 
cmd += f'--fixed_first -K 6 -v 1 --covar_matrix {cmat[0]} '
cmd += f'--WLgamma 0.5 --trait_names {scriptdir}dummy_names.txt'
cmd += f' --outdir {outdir}'